In [ ]:
import pandas as pd
import re 
import gc
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import itertools

from tqdm import tqdm
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score


In [ ]:
pd.set_option('display.max_colwidth', None)
# nltk.download('vader_lexicon')

In [ ]:
# Filtering Work Experience related tweets
def filter_companytweets(company):
    file_path = './ProcessedData/JoinedPeriodData/final_' + company + '_tweets.csv'
    company_tweets = pd.read_csv(file_path, lineterminator='\n')
    print('Read - ', company_tweets.shape)
    
    # Query for extracting tweets related to companies and Work experience
    query = ["work"]
    company_tweets['query_word'] = company_tweets['text'].apply(lambda text: all(re.search(r'\b{}\b'.format(re.escape(word)), text.lower()) for word in query))
    # company_tweets['query_word'] = company_tweets['text'].apply(lambda text: any(word in text.lower() for word in query))
    company_experience_tweets = company_tweets[company_tweets['query_word']]
    
    # Drop duplicates
    print('droping duplicates')
    company_experience_tweets = company_experience_tweets.drop_duplicates(subset=['cleaned_tweet'])
    company_experience_tweets = company_experience_tweets.drop("query_word", axis=1)
    
    print('Extracted tweets related to companies and Work experience - ',company_experience_tweets.shape)
    
    print(company, 'filtered')
    return company_experience_tweets

In [ ]:
# Initialize VADER SentimentIntensityAnalyzer
lexicon_classifier = SentimentIntensityAnalyzer()

# Function to calculate polarity
def classify_sentiment(text):
    text = eval(text)
    text = ' '.join(text)
    scores = lexicon_classifier.polarity_scores(text)
    
    if scores['compound'] > 0:
        return 'positive'
    elif scores['compound'] < 0:
        return 'negative'
    else:
        return 'neutral'

In [ ]:
def sentiment_analysis(company, model, feature_extraction):
    data =  company["tokenized_tweet"]
    target = company["lexicon_sentiment"]
    
    random_states = [10,20,30,40,42,50,60,70,80,100]
    accuracy_list = []
    positive_f1_scores = []
    negative_f1_scores = []
    
    for random_state in random_states:
        # print("Random State:", random_state)
        # Split data into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=random_state)

        # Convert text to features
        X_train_vec = feature_extraction.fit_transform(X_train)
        X_test_vec = feature_extraction.transform(X_test)

        # Train model
        model.fit(X_train_vec, y_train)

        # Predictions on test data
        y_pred = model.predict(X_test_vec)

        accuracy = accuracy_score(y_test, y_pred)
        # print("Accuracy of: ",random_state,"is ", accuracy)
        accuracy_list.append(accuracy)
        
        positive_f1 = f1_score(y_test, y_pred, pos_label='positive')
        positive_f1_scores.append(positive_f1)
        
        # Calculate F1 score for negative class
        negative_f1 = f1_score(y_test, y_pred, pos_label='negative')
        negative_f1_scores.append(negative_f1)
        
        # print()
        
    mean_accuracy = np.mean(accuracy_list).astype(float)
    std_accuracy = np.std(accuracy_list).astype(float)
    
    print("\nMean Accuracy:", mean_accuracy)
    print("Standard Deviation of Accuracy:", std_accuracy)
    
     # Create DataFrame with random state and accuracy values
    result_df = pd.DataFrame({"Random State": random_states, "Accuracy": accuracy_list})
    # Format accuracy values with two decimal points
    result_df["Accuracy"] = result_df["Accuracy"].apply(lambda x: '{:.2f}'.format(x))
    
    f1_scores_df = pd.DataFrame({'Random State': random_states,
                             'Positive F1 Score': positive_f1_scores,
                             'Negative F1 Score': negative_f1_scores})
    
    # Create bar chart
    plt.figure(figsize=(10, 3))
    plt.bar(random_states, accuracy_list, color='skyblue')
    plt.xlabel('Random State')
    plt.ylabel('Accuracy')
    plt.title('Accuracy for Different Random States')
    plt.xticks(random_states)
    plt.grid(axis='y', linestyle='--', alpha=0.8)
    
    plt.text(max(random_states) * 0.95, max(accuracy_list) * 0.95, f'Mean Accuracy: {mean_accuracy:.2f}', fontsize=10, color='red', ha='right')
    plt.text(max(random_states) * 0.95, (max(accuracy_list) * 0.90) - 0.05, f'Std Dev: {std_accuracy:.2f}', fontsize=10, color='green', ha='right')
    
    plt.show()
    
    return result_df, f1_scores_df

In [ ]:
BoW = CountVectorizer()  # Using Bag-of-Words (BoW) model
TF_IDF = TfidfVectorizer()  # Using TF-IDF model

model_LR = LogisticRegression(max_iter=4000)
model_NB = BernoulliNB()

In [ ]:
# Load the company tweets data
# Filter the tweets related to work experience
google_works = filter_companytweets("google")
google_works.to_csv('./ProcessedData/extractedWorkExperience/google_works1.csv', index=False)
print()

microft_works = filter_companytweets("microsoft")
microft_works.to_csv('./ProcessedData/extractedWorkExperience/microsoft_works1.csv', index=False)
print()

apple_works = filter_companytweets("apple")
apple_works.to_csv('./ProcessedData/extractedWorkExperience/apple_works1.csv', index=False)
print()

In [ ]:
# After checking, filtering and doing my groundtruth
# Load final dataset
google = pd.read_csv('./ProcessedData/google_Finaltweets.csv', lineterminator='\n')
apple = pd.read_csv('./ProcessedData/apple_Finaltweets.csv', lineterminator='\n')
microsoft = pd.read_csv('./ProcessedData/microsoft_Finaltweets.csv', lineterminator='\n')

In [ ]:
google.shape, apple.shape, microsoft.shape

In [ ]:
google.head()

In [ ]:
# Appplying Lexicon sentiment classification
companies = [google, apple, microsoft]
for company in tqdm(companies):
    company['lexicon_sentiment'] = company['tokenized_tweet'].apply(classify_sentiment)

In [ ]:
# Calculate accuracy
def vader_accuracy(company):
    ground_truth_labels = company['ground_truth_sentiment']
    predicted_sentiments = company['lexicon_sentiment'] 

    accuracy = accuracy_score(ground_truth_labels, predicted_sentiments)
    print("Accuracy:", accuracy)
    print()
    return accuracy

In [ ]:
print("Google Accuracy")
google_accuracy = vader_accuracy(google)

print("Apple Accuracy")
apple_accuracy = vader_accuracy(apple)

print("Microsoft Accuracy")
microsoft_accuracy = vader_accuracy(microsoft)

In [ ]:
# Calculate counts of agreement and disagreement
def level_of_agreement(company):
    ground_truth_labels = company['ground_truth_sentiment']
    predicted_sentiments = company['lexicon_sentiment'] 
    
    agreement_count = sum(1 for true_label, lexicon_label in zip(ground_truth_labels, predicted_sentiments) if true_label == lexicon_label)
    disagreement_count = len(ground_truth_labels) - agreement_count
    
    agreement_counts = {'Agreement': agreement_count, 'Disagreement': disagreement_count}
    
    return agreement_counts

In [ ]:
# Create subplots for three companies
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
companyNames = ['Google', 'Apple', 'Microsoft']

for i, (company, companyName) in enumerate(zip([google, apple, microsoft], companyNames)):
    agreement_counts = level_of_agreement(company)
    axs[i].pie(agreement_counts.values(), labels=agreement_counts.keys(), autopct='%1.1f%%', explode=(0.1, 0.1), shadow=True, startangle=140, colors=['skyblue', 'lightcoral'])
    axs[i].set_title(f'{companyName} Agreement and Disagreement')

plt.show()

In [ ]:
def sentiment_wordcloud(company):
    sentiment_labels = company['lexicon_sentiment']
    token_words = company['tokenized_tweet'].apply(eval)
    
    flat_tokens = list(itertools.chain.from_iterable(token_words))
    
    # Filter token words based on sentiment labels
    positive_words = [word for word, sentiment in zip(flat_tokens, sentiment_labels) if sentiment == 'positive']
    negative_words = [word for word, sentiment in zip(flat_tokens, sentiment_labels) if sentiment == 'negative']

    # Create WordCloud objects for positive and negative sentiment words
    positive_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(positive_words))
    negative_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(negative_words))

    # Plot WordClouds
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(positive_wordcloud, interpolation='bilinear')
    plt.title('Positive Sentiment Word Cloud')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(negative_wordcloud, interpolation='bilinear')
    plt.title('Negative Sentiment Word Cloud')
    plt.axis('off')

    plt.show()

In [ ]:
sentiment_wordcloud(google)

In [ ]:
print('Apple Work')
final_result = pd.DataFrame()
all_result = []
print("Sentiment Analysis with Logistic Regression and BOW ")
all_result.append(sentiment_analysis(apple,model_LR, BoW))
print("Sentiment Analysis with Logistic Regression and TF-IDF: ")
all_result.append(sentiment_analysis(apple,model_LR, TF_IDF))
print("Sentiment Analysis with Naives Bayes and BOW: ")
all_result.append(sentiment_analysis(apple,model_NB, BoW))
print("Sentiment Analysis with Naives Bayes and TF-IDF: ")
all_result.append(sentiment_analysis(apple,model_NB, TF_IDF))

final_result = pd.concat(all_result, axis=1)
final_result.drop('Random State', axis=1, inplace=True)
final_result['Random State'] = [10,20,30,40,42,50,60,70,80,100]
final_result.columns = ['LR_BOW', 'LR_TFIDF', 'NB_BOW', 'NB_TFIDF', 'Random_state']


In [ ]:
final_result.head(10)

In [ ]:
google_result = pd.DataFrame()
all_g_result = []

# Perform sentiment analysis with CountVectorizer
print('Google Work')
print("Sentiment Analysis with Logistic Regression and BOW ")
all_g_result.append(sentiment_analysis(google,model_LR, BoW))
print("Sentiment Analysis with Logistic Regression and TF-IDF: ")
all_g_result.append(sentiment_analysis(google,model_LR, TF_IDF))
print("Sentiment Analysis with Naives Bayes and BOW: ")
all_g_result.append(sentiment_analysis(google,model_NB, BoW))
print("Sentiment Analysis with Naives Bayes and TF-IDF: ")
all_g_result.append(sentiment_analysis(google,model_NB, TF_IDF))

google_result = pd.concat(all_g_result, axis=1)
google_result.drop('Random State', axis=1, inplace=True)
google_result['Random State'] = [10,20,30,40,42,50,60,70,80,100]
google_result.columns = ['LR_BOW', 'LR_TFIDF', 'NB_BOW', 'NB_TFIDF', 'Random_state']


In [ ]:
google_result.head(10)

In [ ]:
microsoft_result = pd.DataFrame()
all_m_result = []

# Perform sentiment analysis with CountVectorizer
print('Microsoft Work')
print("Sentiment Analysis with Logistic Regression and BOW ")
all_m_result.append(sentiment_analysis(microsoft,model_LR, BoW))
print("Sentiment Analysis with Logistic Regression and TF-IDF: ")
all_m_result.append(sentiment_analysis(microsoft,model_LR, TF_IDF))
print("Sentiment Analysis with Naives Bayes and BOW: ")
all_m_result.append(sentiment_analysis(microsoft,model_NB, BoW))
print("Sentiment Analysis with Naives Bayes and TF-IDF: ")
all_m_result.append(sentiment_analysis(microsoft,model_NB, TF_IDF))

microsoft_result = pd.concat(all_m_result, axis=1)
microsoft_result.drop('Random State', axis=1, inplace=True)
microsoft_result['Random State'] = [10,20,30,40,42,50,60,70,80,100]
microsoft_result.columns = ['LR_BOW', 'LR_TFIDF', 'NB_BOW', 'NB_TFIDF', 'Random_state']

In [ ]:
microsoft_result.head(10)